<a href="https://colab.research.google.com/github/lpsalgueiro1/public/blob/main/teste_python_sql_govendas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash

# Instal Java
apt-get update && apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Install PySpark
pip install -q pyspar

# Install Pandas
pip install -q pandas

#download data
mkdir raw_data
curl https://raw.githubusercontent.com/lpsalgueiro1/public/refs/heads/main/Pedidos.csv -o raw_data/Pedidos.csv

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists...


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
ERROR: Could not find a version that satisfies the requirement pyspar (from versions: none)
ERROR: No matching distribution found for pyspar
mkdir: cannot create directory ‘raw_data’: File exists
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   191  100   191    0     0    677      0 --:--:-- --:--:-- --:--:--   679


In [ ]:
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-8-openjdk-amd64'

from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").getOrCreate()

def cria_tabela(path, nome_tabela):
    df = spark.read \
              .csv(path, header=True)
    df.createOrReplaceTempView(nome_tabela)
    return df

pedidos = cria_tabela("raw_data/Pedidos.csv", "pedidos")


# SQL

**Função para execução de queries**

In [ ]:
import pandas as pd

def q(query, n=30):
    return spark.sql(query).show(n=n, truncate=False)

pedidos = pd.read_csv('raw_data/Pedidos.csv')

Para executar alguma consulta, basta colocar seu código sql dentro da função q como no exemplo abaixo:

```python
q("""
    SELECT *
    FROM vendas
""")
```

e o resultado será exibido na tela. As tabelas são "vendas" e "produtos".

1. Escreva uma consulta SQL para retornar o total de pedidos pagos por cliente, ordenado do maior para o menor total.

In [ ]:
# responda aqui
q("""
  SELECT id_cliente, sum(total_pedido) as total_pedido
  FROM pedidos
  group by id_cliente
  order by total_pedido desc
""")

+----------+------------+
|id_cliente|total_pedido|
+----------+------------+
|101       |550.0       |
|103       |450.0       |
|102       |175.0       |
+----------+------------+



2. Retorne o total geral de vendas (`total_pedido`) apenas dos pedidos com status &#39;Pago&#39;.

In [ ]:
# responda aqui
q("""
SELECT sum(total_pedido) as total_pedido_pago
  FROM pedidos
  where status = 'Pago'
""")

+-----------------+
|total_pedido_pago|
+-----------------+
|750.0            |
+-----------------+



3. Crie uma query que calcule a média do valor dos pedidos para cada cliente (`id_cliente`).

In [ ]:
# responda aqui
q("""
  SELECT id_cliente, avg(total_pedido) as media_valor_pedido
  FROM pedidos
  group by id_cliente
""")

+----------+------------------+
|id_cliente|media_valor_pedido|
+----------+------------------+
|101       |275.0             |
|102       |87.5              |
|103       |450.0             |
+----------+------------------+



4. Escreva uma consulta SQL que recupere os clientes que fizeram mais de 1 pedido pago.

In [ ]:
# responda aqui
q("""
  SELECT id_cliente, count(*) as pedidos
  FROM pedidos
  where status = 'Pago'
  group by id_cliente
  having count(*) > 1
""")

+----------+-------+
|id_cliente|pedidos|
+----------+-------+
+----------+-------+



# Python

5. Escreva um script em Python que faça uma requisição HTTP `GET` para a API fictícia de pedidos `https://api.empresa.com/pedidos` e retorne os pedidos pagos.

In [ ]:
import pandas as pd

url = 'https://raw.githubusercontent.com/lpsalgueiro1/public/refs/heads/main/Pedidos.csv'

df = pd.read_csv(url)

df_pago = df[df['status'].str.contains('Pago')]

print(df_pago)

   id_pedido  id_cliente data_pedido  total_pedido status
0          1         101  2024-02-15           250   Pago
3          4         103  2024-02-18           450   Pago
4          5         102  2024-02-19            50   Pago


6. Construa uma função Python que recebe uma lista de dicionários representando pedidos e retorna a soma dos valores dos pedidos pagos.

In [ ]:
def somaValPago(dataFrm):
  return dataFrm.loc[(dataFrm.status == "Pago") , "total_pedido"].sum()

somaValPago(df)

750

7. Crie um script Python que leia um arquivo CSV chamado `clientes.csv` e crie um
dicionário onde a chave é o `id_cliente` e o valor é um dicionário com os outros campos.

In [ ]:
import pandas as pd

data = pd.read_csv('https://raw.githubusercontent.com/lpsalgueiro1/public/refs/heads/main/Clientes.csv')

data_dict = data.to_dict(orient='records')

print(data_dict)




[{'id_cliente': 1, 'Name': 'Alice', 'Age': 28, 'City': 'New York'}, {'id_cliente': 2, 'Name': 'Bob', 'Age': 32, 'City': 'Los Angeles'}, {'id_cliente': 3, 'Name': 'Charlie', 'Age': 24, 'City': 'Chicago'}]


# Algoritmos e Estruturas de Dados

8. Escreva um algoritmo em Python para encontrar o segundo maior número em uma lista
de números inteiros sem utilizar `max()` ou `sorted()`.

In [ ]:
1# responda aqui

lista = [2,5,94,87,20,34,21,115]

lista.sort()

penult_posicao = len(lista)-1

print(lista[penult_posicao-1])



94


9. Um palíndromo é uma palavra ou número que permanece igual quando lido de trás para frente. Implemente uma função que verifique se um número inteiro é um palíndromo.

In [ ]:
def EUmPalindromo(numeroInteiro):
    return str(numeroInteiro) == str(numeroInteiro)[::-1]

# Testes
print(EUmPalindromo(200))  # False
print(EUmPalindromo(121))  # True


False
True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# FIM!